# 🚗 Speed Estimator
YOLO26 + ByteTrack. Detects when a vehicle crosses a defined red trigger-line, measures its speed at that moment, shows the vehicle type, and displays the reading on-screen.

In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 8.0 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO
import numpy as np
import cv2
from collections import defaultdict, deque
from google.colab import files

model = YOLO("yolo26s.pt")  # swap to yolo26n.pt if you need more speed

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


## Step 1: Upload your video

In [3]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"FPS: {fps}, Resolution: {w}x{h}")

Saving American Highway _travel(1080P_HD).mp4 to American Highway _travel(1080P_HD).mp4
FPS: 30.0, Resolution: 1080x1920


## Step 2: Calibration — grab a sample frame
Open `sample_frame.jpg` after running this. You'll use it to set:
1. `LINE_Y` — the pixel row where your red trigger-line sits
2. The scale (real-world distance per pixel) **at that exact row**, using something with a known size (e.g. a lane-marker dash, US standard 10 ft / 3.05 m, or lane width ≈ 12 ft / 3.66 m)
3. `YELLOW_BOX` — the on-screen rectangle where the speed readout will be drawn

In [4]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 15)
ret, sample_frame = cap.read()
cv2.imwrite("sample_frame.jpg", sample_frame)
print("Saved sample_frame.jpg.")

Saved sample_frame.jpg -- open it and measure your reference points on it.


In [5]:
# ================= FILL THESE IN based on sample_frame.jpg =================

# --- Trigger line ---
LINE_Y = 900                # pixel row of your red line (measure on sample_frame.jpg)

# --- Scale at the line (pick a known real-world distance visible near LINE_Y) ---
REAL_DISTANCE_FT = 12        # e.g. lane width, or a lane-dash length
PIXEL_DISTANCE_AT_LINE = 150 # how many pixels that real distance spans at LINE_Y
FT_PER_PIXEL = REAL_DISTANCE_FT / PIXEL_DISTANCE_AT_LINE

# --- Yellow readout box (x1, y1, x2, y2) in pixel coords ---
YELLOW_BOX = (20, h - 260, 520, h - 40)

# --- Speed measurement window ---
# how many frames of position history to use to compute speed AT the crossing
SPEED_FRAMES = 4

# how long (in frames) to keep showing a reading in the yellow box after a crossing
DISPLAY_HOLD_FRAMES = int(fps * 2.5)   # ~2.5 seconds

print(f"ft per pixel at line: {FT_PER_PIXEL:.4f}")

ft per pixel at line: 0.0800


## Step 3: Vehicle type labels (COCO classes we care about)

In [6]:
VEHICLE_CLASSES = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck",
}

## Step 4: Tracking + line-crossing speed logic

In [7]:
track_history = defaultdict(lambda: deque(maxlen=SPEED_FRAMES + 2))
crossed_ids = set()  # so we only measure each vehicle once

# current readout state (what's shown in the yellow box right now)
current_reading = {"speed": None, "vtype": None, "hold": 0}

def check_crossing_and_measure(track_id, cx, cy, vtype, fps):
    hist = track_history[track_id]
    hist.append((cx, cy))

    if track_id in crossed_ids or len(hist) < 2:
        return None

    prev_y = hist[-2][1]
    curr_y = hist[-1][1]

    crossed = (prev_y < LINE_Y <= curr_y) or (prev_y > LINE_Y >= curr_y)
    if not crossed:
        return None

    crossed_ids.add(track_id)

    # use however many frames of history we have (up to SPEED_FRAMES) to get speed
    pts = list(hist)[-min(len(hist), SPEED_FRAMES):]
    (x1, y1), (x2, y2) = pts[0], pts[-1]
    pixel_dist = np.hypot(x2 - x1, y2 - y1)
    n_frames = len(pts) - 1
    if n_frames == 0:
        return None

    real_dist_ft = pixel_dist * FT_PER_PIXEL
    time_s = n_frames / fps
    speed_fps = real_dist_ft / time_s
    speed_mph = speed_fps * 0.681818

    return speed_mph, vtype

## Step 5: Draw the yellow readout box + overlays

In [8]:
def draw_readout_box(frame, reading):
    x1, y1, x2, y2 = YELLOW_BOX
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 230, 255), 3)

    if reading["speed"] is not None and reading["hold"] > 0:
        text_speed = f'{reading["speed"]:.0f} mph'
        text_type = reading["vtype"]
        cv2.putText(frame, text_speed, (x1 + 20, y1 + 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 230, 255), 4)
        cv2.putText(frame, text_type, (x1 + 20, y1 + 150),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    else:
        cv2.putText(frame, "Waiting for vehicle...", (x1 + 20, y1 + 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 230, 255), 2)
    return frame

## Step 6: Main processing loop

In [9]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
out = cv2.VideoWriter("speed_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

results_gen = model.track(video_path, tracker="bytetrack.yaml", stream=True,
                           classes=list(VEHICLE_CLASSES.keys()))

for result in results_gen:
    frame = result.orig_img.copy()

    # draw the red trigger line
    cv2.line(frame, (0, LINE_Y), (w, LINE_Y), (0, 0, 255), 3)

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls_id in zip(boxes, ids, classes):
            x1, y1, x2, y2 = box
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            vtype = VEHICLE_CLASSES.get(cls_id, "Vehicle")

            color = (0, 200, 0) if track_id not in crossed_ids else (0, 150, 255)
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(frame, f"{vtype} {track_id}", (int(x1), int(y1) - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            result_reading = check_crossing_and_measure(track_id, cx, cy, vtype, fps)
            if result_reading is not None:
                speed_mph, vtype_hit = result_reading
                current_reading["speed"] = speed_mph
                current_reading["vtype"] = vtype_hit
                current_reading["hold"] = DISPLAY_HOLD_FRAMES

    if current_reading["hold"] > 0:
        current_reading["hold"] -= 1

    frame = draw_readout_box(frame, current_reading)
    out.write(frame)

out.release()
print("Done -> speed_output.mp4")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 328ms
Prepared 1 package in 62ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


video 1/1 (frame 1/199) /content/American Highway _travel(1080P_HD).mp4: 640x384 13 cars, 1 truck, 11.8ms
video 1/1 (frame 2/199) /content/American Highway _travel(1080P_HD).mp4: 640x384 13 cars, 1 truck, 12.9ms
video 1/1 (frame 3/199) /content/American Highway _travel(1080P_HD).mp4: 640x384 13 cars, 11.8ms
video 1/1 (frame 4/199) /content/American Highway _travel(1080P_HD).mp4: 640x384 12 cars, 14.4ms
video 1/1 (frame 5/199) /content/American Highway _travel(1080P_HD).mp4: 640x384 12 cars, 14.0ms
video 1/1 (frame 6/199) /content/American Highway _travel(1080P_HD).mp4: 640x384 12 cars, 15.5ms
video 1/1 (frame 7/199) /content/American 

## Step 7: Save to Drive

In [10]:
from google.colab import drive
drive.mount('/content/drive')
!cp speed_output.mp4 /content/drive/MyDrive/speed_output.mp4

Mounted at /content/drive
